<a href="https://colab.research.google.com/github/AbdullahRasheed452/ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [22]:
# Finding 1: Growing content is longer, younger, and ranks slightly better than declining content

# My question: trend direction is based on a recent 30 day change, so
# growing pages may look longer and younger simply because they were
# just refreshed, not because length and age caused the growth

In [23]:
# Finding 2: Weighted CTR drops sharply from top 3 positions to deep positions

# My question: this is a portfolio wide average, total clicks over total
# impressions, so a few very high traffic pages could be pulling the
# number up. A per page median CTR would show if this holds for most
# pages or just a few large ones

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [24]:
import os, subprocess

REPO_URL = "https://github.com/AbdullahRasheed452/ML-Internship"
REPO_DIR = "ML-Internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if os.getcwd().split("/")[-1] != REPO_DIR:
    os.chdir(REPO_DIR)

In [25]:
%pip -q install -U duckdb huggingface_hub

In [26]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF_TOKEN')

In [27]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

In [28]:
q = f"""
SELECT
    f.content_hash_id,
    f.client_hash_id,
    AVG(CASE WHEN f.report_date < '2026-03-16' THEN f.gsc_impressions END) AS avg_impressions_first_half,
    AVG(CASE WHEN f.report_date < '2026-03-16' THEN f.gsc_avg_position END) AS avg_position_first_half,
    AVG(CASE WHEN f.report_date < '2026-03-16' AND f.gsc_impressions > 0
             THEN f.gsc_clicks * 1.0 / f.gsc_impressions END) AS avg_ctr_first_half,
    ANY_VALUE(c.word_count) AS word_count,
    ANY_VALUE(DATE_DIFF('day', c.content_created_date, DATE '2026-03-15')) AS content_age_days,
    SUM(CASE WHEN f.report_date < '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
    SUM(CASE WHEN f.report_date >= '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_second_half
FROM {TABLES['fact_daily']} f
JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
GROUP BY f.content_hash_id, f.client_hash_id
HAVING imp_first_half >= 10
"""

model_data = con.sql(q).df()
model_data['pct_change'] = (model_data['imp_second_half'] - model_data['imp_first_half']) / model_data['imp_first_half']
model_data['is_declining'] = (model_data['pct_change'] < -0.2).astype(int)
model_data = model_data.dropna(subset=['avg_impressions_first_half', 'avg_position_first_half', 'avg_ctr_first_half', 'word_count', 'content_age_days'])

features = ['avg_impressions_first_half', 'avg_position_first_half', 'avg_ctr_first_half', 'word_count', 'content_age_days']
X = model_data[features]
y = model_data['is_declining']

import numpy as np
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print(len(model_data), 'pages,', model_data['client_hash_id'].nunique(), 'clients')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

79616 pages, 41 clients


In [29]:
# Before: a plain random split, ignoring which client each page belongs to
# This risks the same client showing up in both train and test
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.3, random_state=42
)

rf_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_random.fit(X_train_r, y_train_r)

random_scores = rf_random.predict_proba(X_test_r)[:, 1]

for k in (20, 50):
    print(f"Precision@{k} (random split): {precision_at_k(random_scores, y_test_r.values, k):.3f}")

Precision@20 (random split): 0.900
Precision@50 (random split): 0.880


In [30]:
# Before vs after honest split

# Before (random split, ignores client): Precision@20 = 0.900, Precision@50 = 0.920
# After (grouped by client, from Week 5): Precision@20 = 0.650, Precision@50 = 0.700

# The random split looks much better, but this is misleading
# Since the same client can appear in both train and test, the model can
# partly memorize that client's specific patterns instead of learning
# something that generalizes to a brand new client
# The grouped split is the honest number, since it tests the model only
# on clients it never saw during training
# This is a clear example of why split design matters as much as the model itself

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [31]:
# Leakage audit on the final features:
# avg_impressions_first_half, avg_position_first_half, avg_ctr_first_half,
# word_count, content_age_days

# All five only use data from before the label window, so they are safe
# The one real leak I found earlier: avg_impressions used the whole month,
# including the second half the label is built from, giving a fake precision near 1.0
# Fixing it to first half only dropped the score to a believable 0.65 to 0.70, the honest result

# No product flags like health_score were used, and no future dates
# beyond the feature window were used anywhere

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [32]:
# Original claim (Week 5): "The model clearly beats the baseline at both cutoffs"
# This sounds more certain than the evidence supports

# Rewrite: On this one client-grouped test split, the model showed
# higher Precision@20 and Precision@50 than the baseline rule
# This is an observed, directional result on one split of one month
# of data, not a guarantee that the model will always outperform the
# baseline on new data or other time periods

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.